<a href="https://colab.research.google.com/github/opendatas2017/NMC/blob/main/NMCourse_fast_gpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ускорение


### GPU



In [ ]:
import numpy as np
import sys
import time, timeit
print("Numpy version is ",np.__version__)  # Должно показать версию, например 2.0.2

print("Python version is ",sys.version)

Numpy version is  2.0.2
Python version is  3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]


In [ ]:
!pip install cupy-cuda12x -q


In [ ]:
# --- Импорт библиотеки CuPy ---
# CuPy — это drop-in замена NumPy для GPU NVIDIA
# Синтаксис почти идентичен, но вычисления выполняются на видеокарте
import cupy as cp

# --- Создание массива на GPU ---
# cp.arange() работает как np.arange(), но память выделяется на видеокарте
x = cp.arange(5)  # Массив [0, 1, 2, 3, 4] на GPU

# --- Проверка устройства ---
# x.device показывает, на каком GPU размещён массив
# CUDA:0 — первая видеокарта, CUDA:1 — вторая и т.д.
print(f"Массив: {x}")
print(f"Устройство: {x.device}")

# --- Важное замечание ---
# Если у вас нет GPU NVIDIA или не установлен CUDA, эта ячейка выдаст ошибку
# В Google Colab: Среда выполнения → Тип ускорителя → GPU

Массив: [0 1 2 3 4]
Устройство: <CUDA Device 0>


### 💡 Вывод
**CuPy — это «NumPy для GPU» с минимальным порогом входа**

| Характеристика | NumPy | CuPy |
|----------------|-------|------|
| **Устройство** | CPU | GPU (NVIDIA) |
| **Синтаксис** | `np.array` | `cp.array` |
| **Память** | ОЗУ (RAM) | Видеопамять (VRAM) |
| **Ускорение** | 1× (база) | ×10–100 для векторных операций |

**Ключевые моменты:**  
1. 🖥️ **`x.device`** — показывает, на каком GPU лежит массив (CUDA:0, CUDA:1...)  
2. 🔄 **Передача данных** — копирование CPU↔GPU имеет накладные расходы, минимизируйте его  
3. ⚠️ **Требования** — нужна видеокарта NVIDIA + драйверы CUDA + `pip install cupy-cudaXX`  



In [ ]:
import numpy as np
import cupy as cp
import time

# --- Параметры теста ---
# Размер системы: n × n матрица
n = 5000

# ⚠️ ВАЖНО: Приводим к единому типу данных!
# NumPy по умолчанию использует float64, CuPy — float32
# Для честного сравнения точности используем float64 на обоих устройствах
DTYPE = np.float64

print(f"📊 Размер системы: {n} × {n}")
print(f"🔢 Тип данных: {DTYPE.__name__}")
print("=" * 60)

# ================= 1. CPU: Генерация и решение =================
# Данные создаются в ОЗУ, вычисления на процессоре
A_cpu = np.random.randn(n, n).astype(DTYPE)
b_cpu = np.random.randn(n).astype(DTYPE)

t0 = time.perf_counter()
x_cpu = np.linalg.solve(A_cpu, b_cpu)  # LU-разложение на CPU
t_cpu = time.perf_counter() - t0

# Проверка точности: невязка ||Ax - b||
res_cpu = np.linalg.norm(A_cpu @ x_cpu - b_cpu)

print(f"\n🖥️  CPU (NumPy):")
print(f"   Время решения      = {t_cpu:.4f} сек")
print(f"   Норма невязки      = {res_cpu:.3e}")

# ================= 2. GPU: Копирование данных =================
# Передача данных из ОЗУ в видеопамять (PCIe-шина)
t0 = time.perf_counter()
A_gpu = cp.asarray(A_cpu, dtype=cp.float64)  # Явно указываем float64
b_gpu = cp.asarray(b_cpu, dtype=cp.float64)
cp.cuda.Stream.null.synchronize()  # Ждём завершения копирования
t_copy_to_gpu = time.perf_counter() - t0

print(f"\n📤 Копирование CPU → GPU = {t_copy_to_gpu:.4f} сек")

# ================= 3. GPU: Решение на видеокарте =================
# Вычисления выполняются на CUDA-ядрах GPU
t0 = time.perf_counter()
x_gpu = cp.linalg.solve(A_gpu, b_gpu)  # LU-разложение на GPU (cuSOLVER)
cp.cuda.Stream.null.synchronize()  # Ждём завершения вычислений
t_gpu_solve = time.perf_counter() - t0

# Невязка считается на GPU, потом конвертируется для сравнения
res_gpu = cp.linalg.norm(cp.matmul(A_gpu, x_gpu) - b_gpu)
res_gpu_value = float(cp.asnumpy(res_gpu))

print(f"\n🎮 GPU (CuPy):")
print(f"   Время решения      = {t_gpu_solve:.4f} сек")
print(f"   Норма невязки      = {res_gpu_value:.3e}")

# ================= 4. GPU: Возврат результата =================
# Передача данных из видеопамяти обратно в ОЗУ
t0 = time.perf_counter()
x_gpu_to_cpu = cp.asnumpy(x_gpu)
t_copy_to_cpu = time.perf_counter() - t0

print(f"📥 Копирование GPU → CPU = {t_copy_to_cpu:.4f} сек")

# ================= 5. Итоговая статистика =================
t_gpu_total = t_copy_to_gpu + t_gpu_solve + t_copy_to_cpu

print("\n" + "=" * 60)
print(f"📈 СРАВНЕНИЕ ПРОИЗВОДИТЕЛЬНОСТИ:")
print(f"   Ускорение (только solve)     = {t_cpu / t_gpu_solve:.2f}×")
print(f"   Ускорение (с учётом копий)   = {t_cpu / t_gpu_total:.2f}×")

# Проверка согласованности результатов
x_diff = np.linalg.norm(x_cpu - x_gpu_to_cpu) / np.linalg.norm(x_cpu)
print(f"\n🎯 Относительная разница решений = {x_diff:.3e}")

📊 Размер системы: 5000 × 5000
🔢 Тип данных: float64

🖥️  CPU (NumPy):
   Время решения      = 1.8132 сек
   Норма невязки      = 1.446e-09

📤 Копирование CPU → GPU = 0.1973 сек

🎮 GPU (CuPy):
   Время решения      = 1.5522 сек
   Норма невязки      = 2.091e-09
📥 Копирование GPU → CPU = 0.0005 сек

📈 СРАВНЕНИЕ ПРОИЗВОДИТЕЛЬНОСТИ:
   Ускорение (только solve)     = 1.17×
   Ускорение (с учётом копий)   = 1.04×

🎯 Относительная разница решений = 1.903e-11


In [ ]:
import numpy as np
import cupy as cp
import time

# --- Параметры теста ---
# Большое количество элементов для загрузки GPU
# Попробуйте: 10**7, 5*10**7, 10**8 для разных эффектов
n = 10_000_000

print(f"📊 Число элементов: n = {n:,}")
print("=" * 60)

# ⚠️ ВАЖНО: Используем float32 для GPU
# Это родной тип для большинства GPU-операций (быстрее и меньше памяти)
DTYPE = np.float32

# ================= 1. CPU: Вычисления на процессоре =================
# Генерация данных в ОЗУ
x_cpu = np.random.rand(n).astype(DTYPE)

# "Тяжёлая" функция: f(x) = sin(x) + cos(x²)
# Много тригонометрических операций — хорошая нагрузка для сравнения
t0 = time.perf_counter()
y_cpu = np.sin(x_cpu) + np.cos(x_cpu**2)
t_cpu = time.perf_counter() - t0

print(f"\n🖥️  CPU (NumPy):")
print(f"   Время вычисления = {t_cpu:.4f} сек")

# ================= 2. GPU: Копирование на видеокарту =================
# Передача данных через PCIe-шину в видеопамять
t0 = time.perf_counter()
x_gpu = cp.asarray(x_cpu)  # Копирование CPU → GPU
cp.cuda.Stream.null.synchronize()  # Ждём завершения передачи
t_copy_to_gpu = time.perf_counter() - t0

print(f"\n📤 Копирование CPU → GPU = {t_copy_to_gpu:.4f} сек")

# ================= 3. GPU: Вычисления на видеокарте =================
# Та же функция, но выполняется на тысячах CUDA-ядрах параллельно
t0 = time.perf_counter()
y_gpu = cp.sin(x_gpu) + cp.cos(x_gpu**2)  # Массово-параллельные вычисления
cp.cuda.Stream.null.synchronize()  # Ждём завершения всех ядер
t_gpu_compute = time.perf_counter() - t0

print(f"\n🎮 GPU (CuPy):")
print(f"   Время вычисления = {t_gpu_compute:.4f} сек")

# ================= 4. GPU: Возврат результата =================
# Передача результатов обратно в ОЗУ для проверки
t0 = time.perf_counter()
y_gpu_cpu = cp.asnumpy(y_gpu)  # Копирование GPU → CPU
t_copy_to_cpu = time.perf_counter() - t0

print(f"📥 Копирование GPU → CPU = {t_copy_to_cpu:.4f} сек")

# ================= 5. Итоговая статистика =================
t_gpu_total = t_copy_to_gpu + t_gpu_compute + t_copy_to_cpu

# Проверка точности: результаты должны совпадать в пределах float32
max_diff = np.max(np.abs(y_cpu - y_gpu_cpu))

print("\n" + "=" * 60)
print(f"📈 СРАВНЕНИЕ ПРОИЗВОДИТЕЛЬНОСТИ:")
print(f"   Ускорение (только вычисления)   = {t_cpu / t_gpu_compute:.2f}×")
print(f"   Ускорение (с учётом копий)      = {t_cpu / t_gpu_total:.2f}×")
print(f"\n🎯 Максимальное отличие результатов = {max_diff:.3e}")

📊 Число элементов: n = 10,000,000

🖥️  CPU (NumPy):
   Время вычисления = 0.0563 сек

📤 Копирование CPU → GPU = 0.0118 сек

🎮 GPU (CuPy):
   Время вычисления = 0.0020 сек
📥 Копирование GPU → CPU = 0.0144 сек

📈 СРАВНЕНИЕ ПРОИЗВОДИТЕЛЬНОСТИ:
   Ускорение (только вычисления)   = 27.75×
   Ускорение (с учётом копий)      = 2.00×

🎯 Максимальное отличие результатов = 2.384e-07


### 💡 Вывод
**GPU даёт максимальное ускорение для массовых поэлементных операций**

| Характеристика | CPU (NumPy) | GPU (CuPy) |
|----------------|-------------|------------|
| **Архитектура** | 4–16 ядер | 1000–5000 CUDA-ядер |
| **Параллелизм** | Низкий | Массовый (SIMD) |
| **Тип данных** | float64 по умолчанию | float32 эффективнее |
| **Ускорение** | 1× (база) | ×5–50 для векторных операций |

**Ключевые моменты:**  
1. 🎯 **Накладные расходы копирования** — могут составлять 30–70% от общего времени GPU-пути  
2. 📊 **Порог окупаемости** — для n < 10⁶ копирование может «съесть» всю выгоду  
3. 🔢 **float32 vs float64** — на GPU float32 значительно быстрее (меньше памяти, больше пропускная способность)  

**Когда GPU эффективен:**  
✅ Большие массивы (n > 10⁷)  
✅ Множество операций без копирования (данные остаются на GPU)  
✅ Массово-параллельные функции (sin, cos, exp, матричные операции)  

❌ **Не выгодно:** малые данные, сложные ветвления, частое CPU↔GPU копирование  

👉 *Правило: Если данные уже на GPU — выполняйте там ВСЕ возможные вычисления. Копируйте только финальный результат.*

In [ ]:
import numpy as np
import torch
import time

# --- Параметры теста ---
# Размер матрицы: n × n
# Попробуйте: 2000, 4000, 8000 для разных эффектов
n = 3000

# --- Выбор устройства: автоматическое переключение CPU/GPU ---
# PyTorch сам определит доступность CUDA и выберет оптимальное устройство
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"🔧 Устройство PyTorch: {device}")
print(f"📊 Размер матриц: {n} × {n}")
print("=" * 60)

# ⚠️ ВАЖНО: float32 — оптимальный тип для GPU
# float64 работает, но медленнее и занимает больше памяти
DTYPE = np.float32

# ================= 1. NumPy (CPU): Базовое измерение =================
# Данные создаются в ОЗУ, вычисления на процессоре
A_np = np.random.randn(n, n).astype(DTYPE)
B_np = np.random.randn(n, n).astype(DTYPE)

# Матричное умножение: C = A × B
t0 = time.perf_counter()
C_np = A_np @ B_np  # BLAS-библиотека на CPU
t_cpu = time.perf_counter() - t0

print(f"\n🖥️  NumPy (CPU):")
print(f"   Время матричного умножения = {t_cpu:.4f} сек")

# ================= 2. PyTorch (GPU): Копирование данных =================
# Конвертация NumPy массивов в PyTorch тензоры + передача на GPU
t0 = time.perf_counter()
A_t = torch.from_numpy(A_np).to(device)  # CPU → GPU (если доступен)
B_t = torch.from_numpy(B_np).to(device)

# Синхронизация: ждём завершения всех CUDA-операций
if device.type == "cuda":
    torch.cuda.synchronize()

t_copy_to_gpu = time.perf_counter() - t0

print(f"\n📤 Копирование CPU → {device} = {t_copy_to_gpu:.4f} сек")

# ================= 3. PyTorch (GPU): Вычисления на видеокарте =================
# Матричное умножение выполняется на CUDA-ядрах (cuBLAS библиотека)
t0 = time.perf_counter()
C_t = A_t @ B_t  # Массово-параллельное умножение на GPU

if device.type == "cuda":
    torch.cuda.synchronize()  # Важно: без этого замер будет некорректным!

t_torch_compute = time.perf_counter() - t0

print(f"\n🎮 PyTorch ({device}):")
print(f"   Время матричного умножения = {t_torch_compute:.4f} сек")

# ================= 4. PyTorch: Возврат результата =================
# Конвертация тензора обратно в NumPy массив для сравнения
t0 = time.perf_counter()
C_t_cpu = C_t.cpu().numpy()  # GPU → CPU
t_copy_to_cpu = time.perf_counter() - t0

print(f"📥 Копирование {device} → CPU = {t_copy_to_cpu:.4f} сек")

# ================= 5. Итоговая статистика =================
t_torch_total = t_copy_to_gpu + t_torch_compute + t_copy_to_cpu

# Проверка точности: результаты должны совпадать в пределах float32
max_diff = np.max(np.abs(C_np - C_t_cpu))

print("\n" + "=" * 60)
print(f"📈 СРАВНЕНИЕ ПРОИЗВОДИТЕЛЬНОСТИ:")
print(f"   Ускорение (только вычисления)  = {t_cpu / t_torch_compute:.2f}×")
print(f"   Ускорение (с учётом копий)     = {t_cpu / t_torch_total:.2f}×")
print(f"\n🎯 Максимальное отличие результатов = {max_diff:.3e}")

🔧 Устройство PyTorch: cuda
📊 Размер матриц: 3000 × 3000

🖥️  NumPy (CPU):
   Время матричного умножения = 0.7703 сек

📤 Копирование CPU → cuda = 0.0374 сек

🎮 PyTorch (cuda):
   Время матричного умножения = 0.0198 сек
📥 Копирование cuda → CPU = 0.0570 сек

📈 СРАВНЕНИЕ ПРОИЗВОДИТЕЛЬНОСТИ:
   Ускорение (только вычисления)  = 38.83×
   Ускорение (с учётом копий)     = 6.74×

🎯 Максимальное отличие результатов = 4.120e-04


### 💡 Вывод
**PyTorch — фреймворк для глубокого обучения с отличной GPU-поддержкой**

| Характеристика | NumPy | CuPy | PyTorch |
|----------------|-------|------|---------|
| **Назначение** | Общие вычисления | NumPy для GPU | Deep Learning |
| **GPU-бэкенд** | ❌ Нет | ✅ CUDA | ✅ CUDA/CPU |
| **Автодифф** | ❌ Нет | ❌ Нет | ✅ Есть |
| **Экосистема** | Большая | Средняя | Огромная |

**Ключевые моменты:**  
1. 🔄 **`torch.cuda.synchronize()`** — обязателен для корректного замера времени на GPU!  
2. 📊 **`.to(device)`** — универсальный способ переноса тензора на CPU/GPU  
3. 🎯 **`.cpu().numpy()`** — конвертация тензора обратно в NumPy массив  

**Преимущества PyTorch:**  
✅ Автодифференцирование (градиенты для обучения моделей)  
✅ Отличная поддержка GPU через cuBLAS/cuDNN  
✅ Гибкая работа с динамическими графами вычислений  

**Недостатки:**  
❌ Избыточен для простых численных задач (лучше CuPy)  
❌ Больше накладных расходов на создание тензоров  



In [ ]:
import numpy as np
import torch
import time

# --- Параметры теста ---
# Размер матрицы: n × n
# Попробуйте: 2000, 4000, 8000 для разных эффектов
n = 3000

# --- Выбор устройства: автоматическое переключение CPU/GPU ---
# PyTorch сам определит доступность CUDA и выберет оптимальное устройство
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"🔧 Устройство PyTorch: {device}")
print(f"📊 Размер матриц: {n} × {n}")
print("=" * 60)


# float64 работает, но медленнее и занимает больше памяти
DTYPE = np.float64

# ================= 1. NumPy (CPU): Базовое измерение =================
# Данные создаются в ОЗУ, вычисления на процессоре
A_np = np.random.randn(n, n).astype(DTYPE)
B_np = np.random.randn(n, n).astype(DTYPE)

# Матричное умножение: C = A × B
t0 = time.perf_counter()
C_np = A_np @ B_np  # BLAS-библиотека на CPU
t_cpu = time.perf_counter() - t0

print(f"\n🖥️  NumPy (CPU):")
print(f"   Время матричного умножения = {t_cpu:.4f} сек")

# ================= 2. PyTorch (GPU): Копирование данных =================
# Конвертация NumPy массивов в PyTorch тензоры + передача на GPU
t0 = time.perf_counter()
A_t = torch.from_numpy(A_np).to(device)  # CPU → GPU (если доступен)
B_t = torch.from_numpy(B_np).to(device)

# Синхронизация: ждём завершения всех CUDA-операций
if device.type == "cuda":
    torch.cuda.synchronize()

t_copy_to_gpu = time.perf_counter() - t0

print(f"\n📤 Копирование CPU → {device} = {t_copy_to_gpu:.4f} сек")

# ================= 3. PyTorch (GPU): Вычисления на видеокарте =================
# Матричное умножение выполняется на CUDA-ядрах (cuBLAS библиотека)
t0 = time.perf_counter()
C_t = A_t @ B_t  # Массово-параллельное умножение на GPU

if device.type == "cuda":
    torch.cuda.synchronize()  # Важно: без этого замер будет некорректным!

t_torch_compute = time.perf_counter() - t0

print(f"\n🎮 PyTorch ({device}):")
print(f"   Время матричного умножения = {t_torch_compute:.4f} сек")

# ================= 4. PyTorch: Возврат результата =================
# Конвертация тензора обратно в NumPy массив для сравнения
t0 = time.perf_counter()
C_t_cpu = C_t.cpu().numpy()  # GPU → CPU
t_copy_to_cpu = time.perf_counter() - t0

print(f"📥 Копирование {device} → CPU = {t_copy_to_cpu:.4f} сек")

# ================= 5. Итоговая статистика =================
t_torch_total = t_copy_to_gpu + t_torch_compute + t_copy_to_cpu

# Проверка точности: результаты должны совпадать в пределах float32
max_diff = np.max(np.abs(C_np - C_t_cpu))

print("\n" + "=" * 60)
print(f"📈 СРАВНЕНИЕ ПРОИЗВОДИТЕЛЬНОСТИ:")
print(f"   Ускорение (только вычисления)  = {t_cpu / t_torch_compute:.2f}×")
print(f"   Ускорение (с учётом копий)     = {t_cpu / t_torch_total:.2f}×")
print(f"\n🎯 Максимальное отличие результатов = {max_diff:.3e}")

🔧 Устройство PyTorch: cuda
📊 Размер матриц: 3000 × 3000

🖥️  NumPy (CPU):
   Время матричного умножения = 0.9121 сек

📤 Копирование CPU → cuda = 0.0344 сек

🎮 PyTorch (cuda):
   Время матричного умножения = 0.4004 сек
📥 Копирование cuda → CPU = 0.0469 сек

📈 СРАВНЕНИЕ ПРОИЗВОДИТЕЛЬНОСТИ:
   Ускорение (только вычисления)  = 2.28×
   Ускорение (с учётом копий)     = 1.89×

🎯 Максимальное отличие результатов = 1.222e-12
